# P4: SQL Revenue Dashboard

**Module:** 1 — Foundations & Data Engineering  
**Week:** 4  

## Problem Statement
Build a comprehensive revenue analytics dashboard using SQL queries against a relational database. This is the capstone project for Module 1, bringing together Python, SQL, Pandas, and visualization skills.

## Metrics
- Monthly revenue, profit, margin, and MoM growth
- Revenue breakdown by category, region, and tier
- Customer LTV and churn indicators
- Top products and basket analysis

In [ ]:
import sys
sys.path.insert(0, '../../..')

import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from shared.viz_utils import setup_style, save_fig
setup_style()

---
## 1. Set Up the Database

In [ ]:
# Generate the database
from src.setup_db import create_tables, populate_data

conn = sqlite3.connect('data/store.db')
create_tables(conn)
populate_data(conn)

# Verify
for table in ['customers', 'products', 'orders', 'order_items']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"  {table}: {count} rows")

---
## 2. Executive Summary

In [ ]:
from src.queries import executive_summary, monthly_revenue, revenue_by_category, revenue_by_region, top_products, customer_ltv

summary = executive_summary(conn)
print("=" * 60)
print("  EXECUTIVE SUMMARY")
print("=" * 60)
for col in summary.columns:
    val = summary[col].iloc[0]
    if isinstance(val, float) and val > 1000:
        print(f"  {col:25s}: ${val:>15,.2f}")
    else:
        print(f"  {col:25s}: {val:>15}")
print("=" * 60)

---
## 3. Monthly Revenue Trend

In [ ]:
monthly = monthly_revenue(conn)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Revenue & Profit
ax1.plot(monthly['month'], monthly['revenue'], 'b-o', label='Revenue', markersize=4)
ax1.plot(monthly['month'], monthly['profit'], 'g-s', label='Profit', markersize=4)
ax1.fill_between(range(len(monthly)), monthly['profit'], alpha=0.1, color='green')
ax1.set_title('Monthly Revenue & Profit')
ax1.set_ylabel('Amount ($)')
ax1.legend()

# Growth Rate
colors = ['green' if x > 0 else 'red' for x in monthly['growth_pct'].fillna(0)]
ax2.bar(range(len(monthly)), monthly['growth_pct'].fillna(0), color=colors, alpha=0.7)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_title('Month-over-Month Growth Rate (%)')
ax2.set_ylabel('Growth %')
ax2.set_xticks(range(len(monthly)))
ax2.set_xticklabels(monthly['month'], rotation=45, ha='right')

plt.tight_layout()
save_fig(fig, 'outputs/monthly_revenue.png')
plt.show()

---
## 4. Revenue by Category

In [ ]:
by_cat = revenue_by_category(conn)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
ax1.barh(by_cat['category'], by_cat['revenue'], edgecolor='black')
ax1.set_xlabel('Revenue ($)')
ax1.set_title('Revenue by Category')

# Margin comparison
by_cat['margin_pct'] = ((by_cat['revenue'] - by_cat['cost']) / by_cat['revenue'] * 100).round(1)
ax2.barh(by_cat['category'], by_cat['margin_pct'], edgecolor='black', color='green', alpha=0.7)
ax2.set_xlabel('Profit Margin (%)')
ax2.set_title('Profit Margin by Category')

plt.tight_layout()
save_fig(fig, 'outputs/revenue_by_category.png')
plt.show()

---
## 5. Revenue by Region

In [ ]:
by_region = revenue_by_region(conn)

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(by_region['region'], by_region['revenue'], edgecolor='black')
ax.set_title('Revenue by Region')
ax.set_ylabel('Revenue ($)')

# Add value labels
for bar, val in zip(bars, by_region['revenue']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1000,
            f'${val:,.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
save_fig(fig, 'outputs/revenue_by_region.png')
plt.show()

---
## 6. Top Products

In [ ]:
top = top_products(conn, n=10)

fig, ax = plt.subplots(figsize=(10, 6))
colors = plt.cm.Set3(range(len(top)))
ax.barh(top['product'], top['revenue'], color=colors, edgecolor='black')
ax.set_xlabel('Revenue ($)')
ax.set_title('Top 10 Products by Revenue')
ax.invert_yaxis()

plt.tight_layout()
save_fig(fig, 'outputs/top_products.png')
plt.show()

---
## 7. Customer LTV Analysis

In [ ]:
ltv = customer_ltv(conn)

print(f"Total customers: {len(ltv)}")
print(f"\nLTV Distribution:")
print(ltv['lifetime_value'].describe().round(2))

# LTV distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ltv['lifetime_value'].hist(bins=30, ax=ax1, edgecolor='black', alpha=0.7)
ax1.set_title('Customer LTV Distribution')
ax1.set_xlabel('Lifetime Value ($)')
ax1.axvline(ltv['lifetime_value'].median(), color='red', linestyle='--', label='Median')
ax1.legend()

# LTV by tier
ltv.boxplot(column='lifetime_value', by='tier', ax=ax2)
ax2.set_title('LTV by Customer Tier')
ax2.set_ylabel('Lifetime Value ($)')
plt.suptitle('')

plt.tight_layout()
save_fig(fig, 'outputs/customer_ltv.png')
plt.show()

---
## 8. Additional Analyses

TODO: Write your own SQL queries for these analyses.

In [ ]:
# TODO: Revenue heatmap — Month × Category
# Write a SQL query that returns month, category, and revenue
# Pivot in Pandas and visualize with sns.heatmap


In [ ]:
# TODO: Basket analysis
# Write a SQL query that calculates:
# - Average items per order
# - Average unique products per order
# - Average order value
# - How these metrics change by customer tier


In [ ]:
# TODO: Churn indicator
# Identify customers who haven't ordered in the last 90 days
# What % of each tier is at risk?


---
## 9. Dashboard Summary Report

Write a concise executive summary covering:
- Overall performance (revenue, profit, margin)
- Key trends (growth, seasonality)
- Segment insights (region, category, customer tier)
- Risks (churn, declining segments)
- Recommendations (2-3 actionable items)

In [ ]:
# TODO: Write your executive summary


In [ ]:
conn.close()

---
## What I Learned
- 

## What I'd Do Differently
- 